In [ ]:
# Fig. 5: current deduplicated records, city-level descriptive associations.
import json
from pathlib import Path
import matplotlib.pyplot as plt
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data" / "7.匹配时间" / "merged_output.csv").is_file())
QA = ROOT / "检查重复" / "fig5更新_20261109"
QA.mkdir(parents=True, exist_ok=True)
plt.rcParams['font.sans-serif'] = ['Times New Roman', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
df = pd.read_csv(ROOT / "data" / "7.匹配时间" / "merged_output.csv",
                 usecols=["裁判日期", "case_dt_hour", "lng", "lat", "death"], low_memory=False)
run_summary = {"input_records": len(df), "source": "data/7.匹配时间/merged_output.csv",
               "unit": "retained court record; no incident-level merging",
               "period_filter": "none, preserving the original Fig. 5 population"}


In [ ]:
df

# 计算发生时间到判决时间的天数

In [ ]:
# Parse each year-first date format explicitly; never silently lose valid dates.
for column in ["裁判日期", "case_dt_hour"]:
    df[column] = pd.to_datetime(df[column].astype("string").str.strip(),
                                format="mixed", yearfirst=True, errors="coerce")
    run_summary[f"unparsed_{column}"] = int(df[column].isna().sum())
    if df[column].isna().any():
        raise ValueError(f"Review unparsed {column} before producing Fig. 5")
df["judge_time"] = (df["裁判日期"] - df["case_dt_hour"]).dt.days


# 去除掉负数天数

In [ ]:
# Preserve the original exclusion of physically negative intervals.
neg_mask = df["judge_time"].lt(0)
run_summary["negative_intervals_excluded"] = int(neg_mask.sum())
df = df.loc[~neg_mask].copy()
run_summary["after_interval_screening"] = len(df)


In [ ]:
df

# 空间连接到天地图城市

In [ ]:
city_gdf = gpd.read_file(ROOT / "figure" / "china_geodata" / "中国_市_eng.shp")

In [ ]:
city_gdf

In [ ]:
# 转换经纬度

# 3) 只保留需要字段，修一下几何（可选但推荐）
city_gdf = city_gdf[["name_eng", "geometry"]].copy()
city_gdf["geometry"] = city_gdf["geometry"].buffer(0)  # 修复少量无效几何（常见且有效）

# 4) CRS 对齐（你的 geojson 通常是经纬度 WGS84，但有时 crs 为空）
#    点数据一定是 EPSG:4326（经纬度）
if city_gdf.crs is None:
    city_gdf = city_gdf.set_crs(epsg=4326)
else:
    city_gdf = city_gdf.to_crs(epsg=4326)

# 5) 把 lng/lat 转成数值，生成点 GeoDataFrame
df["lng"] = pd.to_numeric(df["lng"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

points_gdf = gpd.GeoDataFrame(
    df[["lng", "lat"]].copy(),
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326")

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# =========================
# 参数区：按需修改
# =========================
LNG_COL = "lng"
LAT_COL = "lat"
CITY_NAME_COL = "name_eng"          # city_gdf 里城市名称字段
OUTPUT_COL = "city_name"        # 写回 df 的列名

PREDICATE = "within"            # "within" 或 "intersects"
USE_NEAREST_FALLBACK = True     # 是否对 NaN 做 nearest 兜底
MAX_NEAREST_DIST_M = None       # e.g. 50000 表示 50km，None 表示不限制

# =========================
# 0) 基础清洗：经纬度有效性
# =========================
df = df.copy()

# 转数值，非法变 NaN
df[LNG_COL] = pd.to_numeric(df[LNG_COL], errors="coerce")
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")

# 可选：过滤明显不合法的经纬度
valid_coord = (
    df[LNG_COL].between(-180, 180, inclusive="both") &
    df[LAT_COL].between(-90, 90, inclusive="both")
)

# =========================
# 1) 构建点 GeoDataFrame（WGS84）
# =========================
points_gdf = gpd.GeoDataFrame(
    df.loc[valid_coord].copy(),
    geometry=gpd.points_from_xy(df.loc[valid_coord, LNG_COL], df.loc[valid_coord, LAT_COL]),
    crs="EPSG:4326"
)

# =========================
# 2) 确保 city_gdf 有 CRS，并与点一致（用于 within/intersects）
# =========================
if city_gdf.crs is None:
    raise ValueError("city_gdf.crs is None：请先给城市面数据设置正确的 CRS（例如 EPSG:4326）。")

# 先把城市面转换到 EPSG:4326，以便和 points_gdf 做空间连接
city_4326 = city_gdf.to_crs(points_gdf.crs)

# =========================
# 3) 空间连接：点落在哪个城市面
# =========================
joined = gpd.sjoin(
    points_gdf,
    city_4326[[CITY_NAME_COL, "geometry"]],
    how="left",
    predicate=PREDICATE
)

# 处理：极少数一个点命中多个面 -> 取第一个
city_name = joined[CITY_NAME_COL].groupby(joined.index).first()

# 先创建输出列
df[OUTPUT_COL] = pd.NA
# 回填到原 df（对齐索引）
df.loc[city_name.index, OUTPUT_COL] = city_name

# =========================
# 4) nearest 兜底：只对仍是 NaN 的点做
# =========================
if USE_NEAREST_FALLBACK:
    mask_na = df[OUTPUT_COL].isna() & valid_coord

    if mask_na.any():
        # 只取需要兜底的点
        points_na = gpd.GeoDataFrame(
            df.loc[mask_na].copy(),
            geometry=gpd.points_from_xy(df.loc[mask_na, LNG_COL], df.loc[mask_na, LAT_COL]),
            crs="EPSG:4326"
        )

        # --- 关键修复：投影到米制 CRS 再 nearest（避免 warning & 距离更准确） ---
        # 自动估算一个适合此区域的 UTM CRS（单位：米）
        proj_crs = city_4326.estimate_utm_crs()

        points_proj = points_na.to_crs(proj_crs)
        city_proj = city_4326.to_crs(proj_crs)

        nearest = gpd.sjoin_nearest(
            points_proj,
            city_proj[[CITY_NAME_COL, "geometry"]],
            how="left",
            distance_col="nearest_dist_m"
        )

        # 如果你想限制最大兜底距离（比如海上点别硬匹配）
        if MAX_NEAREST_DIST_M is not None:
            nearest.loc[nearest["nearest_dist_m"] > MAX_NEAREST_DIST_M, CITY_NAME_COL] = pd.NA

        # --- 关键修复：按 index 对齐回填，避免 .values 顺序错位 ---
        nearest_name = nearest[CITY_NAME_COL].groupby(level=0).first()
        df.loc[nearest_name.index, OUTPUT_COL] = nearest_name
run_summary["invalid_coordinates_excluded_from_city_groups"] = int((~valid_coord).sum())
run_summary["unassigned_city_records"] = int(df[OUTPUT_COL].isna().sum())

run_summary["invalid_coordinates_excluded_from_city_groups"] = int((~valid_coord).sum())
run_summary["unassigned_city_records"] = int(df[OUTPUT_COL].isna().sum())

run_summary["invalid_coordinates_excluded_from_city_groups"] = int((~valid_coord).sum())
run_summary["unassigned_city_records"] = int(df[OUTPUT_COL].isna().sum())

run_summary["invalid_coordinates_excluded_from_city_groups"] = int((~valid_coord).sum())
run_summary["unassigned_city_records"] = int(df[OUTPUT_COL].isna().sum())

run_summary["invalid_coordinates_excluded_from_city_groups"] = int((~valid_coord).sum())
run_summary["unassigned_city_records"] = int(df[OUTPUT_COL].isna().sum())


In [ ]:
df

In [ ]:
# 筛选列
cols = ["death", "judge_time", "city_name"]
df = df[cols]

In [ ]:
df

# 统计缺失值

In [ ]:
# 统计 judge_time 和 death 是否有缺失值，以及缺失多少
cols = ["death", "judge_time"]

missing_cnt = df[cols].isna().sum()               # 每列缺失个数
missing_ratio = df[cols].isna().mean()            # 每列缺失比例(0~1)

missing_summary = pd.DataFrame({
    "missing_count": missing_cnt,
    "missing_ratio": missing_ratio
}).reset_index().rename(columns={"index": "column"})

print(missing_summary)

# 如果你还想看“任意一个字段缺失”的行数：
rows_with_missing = df[cols].isna().any(axis=1).sum()
print("rows_with_missing(death or judge_time):", rows_with_missing)


# 清洗数据

In [ ]:
df = df.copy()

df.loc[:, "death"] = pd.to_numeric(df["death"], errors="coerce")
df.loc[:, "judge_time"] = pd.to_numeric(df["judge_time"], errors="coerce")

In [ ]:
df

# judge_time：算平均值（mean）
# death_1_ratio：算该城市 death==1.0 的占比（= 1.0 的个数 / 该城市总数）
# 保留每个城市的样本量 n

In [ ]:
# Keep all records in the city denominator, as in the original definition.
result = df.groupby("city_name", as_index=False).agg(
    judge_time_mean=("judge_time", "mean"), n=("death", "size"),
    interval_n=("judge_time", "count"),
    missing_death_n=("death", lambda s: s.isna().sum()),
    death_1_cnt=("death", lambda s: s.eq(1).sum()))
assert result["interval_n"].eq(result["n"]).all()
result["judge_time_mean"] = result["judge_time_mean"].round().astype("Int64")
result["death_1_ratio"] = result["death_1_cnt"] / result["n"]
run_summary["cities"] = len(result)
run_summary["records_in_city_groups"] = int(result.n.sum())
run_summary["missing_death_in_city_groups"] = int(result.missing_death_n.sum())
(QA / "run_summary.json").write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
result


# 保留全部有记录的城市，不设置固定城市数量上限

In [ ]:
result_top = result.sort_values("n", ascending=False).reset_index(drop=True)
result_top

# 两个子图及合并图
沿用原有城市记录数、死亡标记比例及平均间隔定义。所有城市均绘制；
回归线为OLS拟合，阴影为拟合均值的95%置信区间。仅标注部分城市以保持可读性。
数值为城市聚合的描述性结果，不解释为事故发生率或司法效率的因果效应。


## 绘制并输出两个子图和左右组合图

In [ ]:
"""Render the two Fig. 5 panels and their side-by-side composition."""
from pathlib import Path
from itertools import product
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path as PlotPath
import statsmodels.api as sm

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Times New Roman', 'DejaVu Sans'],
    'font.size': 10,
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

def label_cities(ax, data, metric, line_x, line_y):
    """Place small labels near their points while clearing ticks and other marks."""
    count = 14 if metric == 'death_1_ratio' else 20
    chosen = data.nlargest(count, 'n')
    special_names = {'KeZiLeiSuKeErKeZi': 'Kizilsu', 'WuZhiShan': 'Wuzhishan'}
    if metric == 'judge_time_mean':
        chosen = data.loc[data.index.isin(chosen.index) | data.city_name.isin(special_names)]

    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    px_per_point = ax.figure.dpi / 72
    points = ax.transData.transform(data[['n', metric]].to_numpy(dtype=float))
    grid_x = ax.transData.transform(np.c_[ax.get_xticks(), np.zeros(len(ax.get_xticks()))])[:, 0]
    grid_y = ax.transData.transform(np.c_[np.zeros(len(ax.get_yticks())), ax.get_yticks()])[:, 1]
    visible = np.isfinite(line_y)
    line = ax.transData.transform(np.c_[line_x[visible], line_y[visible]])
    offsets = sorted(product(range(-44, 45, 4), range(-36, 37, 3)),
                     key=lambda p: (p[0]**2 + p[1]**2, p[1] < 0))
    occupied = []
    leaders = []
    for row in chosen.itertuples(index=False):
        name = special_names.get(row.city_name, row.city_name.title())
        label = ax.annotate(name, (row.n, getattr(row, metric)), xytext=(0, 0),
                            textcoords='offset points', fontsize=8, color='#2b2b2b',
                            ha='center', va='center', zorder=5)
        point = ax.transData.transform(label.xy)
        for offset in offsets:
            label.set_position(offset)
            box = label.get_window_extent(renderer).padded(.9 * px_per_point)
            if not all(ax.bbox.contains(x, y) for x, y in [box.p0, box.p1]):
                continue
            # Leave a readable gap between labels on the same line.
            if any(box.y0 < other.y1 and box.y1 > other.y0
                   and box.x0 < other.x1 + 7 * px_per_point
                   and box.x1 > other.x0 - 7 * px_per_point for other in occupied):
                continue
            if any(leader.intersects_bbox(box, filled=False) for leader in leaders):
                continue
            if np.any((grid_x >= box.x0) & (grid_x <= box.x1)) or np.any((grid_y >= box.y0) & (grid_y <= box.y1)):
                continue
            marker_box = box.padded(2.6 * px_per_point)
            if np.any((points[:, 0] >= marker_box.x0) & (points[:, 0] <= marker_box.x1)
                      & (points[:, 1] >= marker_box.y0) & (points[:, 1] <= marker_box.y1)):
                continue
            x0, x1 = max(box.x0, line[0, 0]), min(box.x1, line[-1, 0])
            if x0 <= x1:
                low, high = sorted(np.interp([x0, x1], line[:, 0], line[:, 1]))
                if low <= box.y1 and high >= box.y0:
                    continue
            # Connect displaced labels to their own points in crowded regions.
            if np.hypot(*offset) >= 13:
                end = np.clip(point, box.p0, box.p1)
                direction = (end - point) / np.linalg.norm(end - point)
                start = point + direction * (2.5 * px_per_point)
                end = end - direction * (.6 * px_per_point)
                leader = PlotPath([start, end])
                if (any(leader.intersects_bbox(other, filled=False) for other in occupied)
                        or any(leader.intersects_path(other, filled=False) for other in leaders)):
                    continue
                xy = ax.transData.inverted().transform([start, end])
                ax.plot(xy[:, 0], xy[:, 1], color='#777777', linewidth=.45, zorder=4)
                leaders.append(leader)
            occupied.append(box)
            break
        else:
            label.remove()
    return len(occupied)


def plot_panel(ax, data, metric, letter):
    """Show all city observations; OLS band is the 95% CI for the mean."""
    data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=['n', metric])
    data = data.loc[data.n.gt(0)].copy()
    x = data.n.to_numpy(dtype=float)
    y = data[metric].to_numpy(dtype=float)
    model = sm.OLS(y, sm.add_constant(x)).fit()
    grid = np.linspace(x.min(), x.max(), 300)
    pred = model.get_prediction(sm.add_constant(grid)).summary_frame(alpha=.05)
    ax.scatter(x, y, s=17, c='#d1e5f5', edgecolors='#33729d', linewidths=.5, zorder=3)
    limits = (0, 1) if metric == 'death_1_ratio' else (100, 660)
    # Make normal axes clipping explicit in vector geometry as well.
    ax.fill_between(grid, np.clip(pred.mean_ci_lower.to_numpy(), *limits),
                    np.clip(pred.mean_ci_upper.to_numpy(), *limits),
                    color='#d62728', alpha=.18, linewidth=0, zorder=1)
    mean = pred['mean'].to_numpy()
    visible = np.where((mean >= limits[0]) & (mean <= limits[1]), mean, np.nan)
    ax.plot(grid, visible, color='#d62728', linewidth=1.2, zorder=2)
    ax.set_xlabel('Number of Included Court-Adjudicated Cases', fontsize=10)
    ax.set_xlim(-2000, 51000)
    ax.set_xticks(np.arange(0, 50001, 10000))
    ax.set_axisbelow(True)
    ax.grid(True, color='#dddddd', linewidth=.55, alpha=.8)
    ax.text(0, 1.025, letter, transform=ax.transAxes, ha='left', va='bottom',
            fontsize=11, fontweight='bold')
    if metric == 'death_1_ratio':
        ax.set_ylabel('Proportion of Included Cases Involving a Fatality', fontsize=10)
        ax.set_ylim(0, 1)
    else:
        ax.set_ylabel('Mean Accident-to-Judgment Interval (days)', fontsize=10)
        ax.set_ylim(100, 660)
        ax.set_yticks(np.arange(100, 601, 100))
    label_count = label_cities(ax, data, metric, grid, visible)
    return {'cities':len(data), 'city_labels':label_count, 'intercept':float(model.params[0]),
            'slope_per_10000_records':float(model.params[1]*10000),
            'r_squared':float(model.rsquared), 'band':'95% confidence interval for OLS mean'}

def export_fig5(data, output, qa):
    output, qa = Path(output), Path(qa)
    output.mkdir(parents=True, exist_ok=True); qa.mkdir(parents=True, exist_ok=True)
    data.to_csv(qa/'city_aggregates.csv', index=False, encoding='utf-8-sig')
    result = {}
    for metric, stem, letter in [('death_1_ratio','case_num_death_ratio_NOLABEL','(a)'),
                                  ('judge_time_mean','case_num_judge_time_NOLABEL','(b)')]:
        fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
        fig.subplots_adjust(left=.17, right=.96, bottom=.16, top=.92)
        result[metric] = plot_panel(ax, data, metric, letter)
        fig.canvas.draw()
        require_matplotlib_panel_alignment(fig, json_out=qa/f'{stem}.alignment.json', strict=True)
        fig.savefig(output/f'{stem}.png', dpi=300)
        # Keep the existing alternate fatality-panel filename synchronized.
        if metric == 'death_1_ratio': fig.savefig(output/'case_num_death_ratio.png', dpi=300)
        fig.savefig(qa/f'{stem}.pdf')
        fig.savefig(qa/f'{stem}.svg')
        plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=300)
    fig.subplots_adjust(left=.085, right=.98, bottom=.16, top=.92, wspace=.30)
    for ax, metric, letter in zip(axes, ['death_1_ratio','judge_time_mean'], ['(a)','(b)']):
        plot_panel(ax, data, metric, letter)
    fig.canvas.draw()
    require_matplotlib_panel_alignment(fig, json_out=qa/'combin.alignment.json', strict=True)
    fig.savefig(output/'combin.png', dpi=300)
    fig.savefig(qa/'combin.pdf')
    fig.savefig(qa/'combin.svg')
    (qa/'regression.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    return fig, result

# Local axes-geometry check; replaces the unavailable external QA utility.
def require_matplotlib_panel_alignment(fig, json_out, strict=True):
    fig.canvas.draw()
    axes = [ax for ax in fig.axes if ax.get_visible()]
    boxes = [ax.get_window_extent().bounds for ax in axes]
    # This notebook exports single panels and a horizontal two-panel composition.
    aligned = len(boxes) <= 1 or all(
        abs(b[1] - boxes[0][1]) <= 0.5 and abs(b[3] - boxes[0][3]) <= 0.5
        for b in boxes[1:])
    report = {"check": "horizontal_axes_bottom_and_height", "tolerance_px": 0.5,
              "aligned": aligned, "axes_bounds_px": boxes}
    Path(json_out).write_text(json.dumps(report, indent=2), encoding="utf-8")
    if strict and not aligned:
        raise ValueError("Figure panels have unequal bottom positions or heights")
    return report


fig5, regression_summary = export_fig5(result_top, ROOT / "figure" / "fig5", QA)
plt.show()
plt.close(fig5)
regression_summary

## 统计核对

In [ ]:
run_summary